In [ ]:
import numpy as np
import cv2
import glob
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 10)

# termination criteria
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TermCriteria_MAX_ITER, 30, 0.001)

# prepare objects points like (0,0,0), (1,0,0), (2,0,0), ... ,(6,5,0)
objp = np.zeros((6 * 8, 3), np.float32)
objp[:, :2] = np.mgrid[0:6, 0:8].T.reshape(-1, 2) * 0.027

# Arrays to store objects points and image from all images
objpoints = [] #coordinates in 3d in real world
imgpoints = [] #coordinates in 2d in image plane

images = glob.glob('calib/*.jpg')

for fname in images:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Find the chess board corners
    ret, corners = cv2.findChessboardCorners(gray, (6, 8), None)
    
    # If found, add objects points, image points (after refining them)
    if ret == True:
        objpoints.append(objp)
        
        corners2 = cv2.cornerSubPix(gray,corners, (11, 11), (-1, -1), criteria)
        imgpoints.append(corners2)
        
        cv2.drawChessboardCorners(img, (6, 8), corners2, ret)

        plt.imshow(img)
        plt.axis('off')
        plt.show()

In [ ]:
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints,
                gray.shape[::-1],None,None)#, flags=cv2.CALIB_TILTED_MODEL)

print("ret", ret)

print("\nmtx", mtx)

print("\ndist", dist)

print("\nrvecs")

for i in range(7):
    print(rvecs[i])
    print("")

print("\ntvecs")

for i in range(7):
    print(tvecs[i])
    print("")

In [ ]:
import numpy as np
import cv2
import glob
import matplotlib.pyplot as plt
import math

A = np.array(mtx)
dist = np.array(dist)

objp = np.zeros((8 * 6, 3), np.float32)
objp[:, :2] = np.mgrid[0:6, 0:8].T.reshape(-1, 2) * 0.027

# Arrays to store objects points and image from all images
objpoints = [] #coordinates in 3d in real world
imgpoints = [] #coordinates in 2d in image plane

filename = 0

cam = cv2.VideoCapture(1, cv2.CAP_DSHOW)
cam.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cam.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)

while(True):
    success, frame = cam.read()
    
    if (success == False):
        
        cam.release()
        cam = cv2.VideoCapture(cv2.CAP_DSHOW)
        cam.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
        cam.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
        continue
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    ret, corners = cv2.findChessboardCorners(gray, (6, 8), None)
    
    if (ret == True):
        s, rvec, tvec = cv2.solvePnP(objp, corners, A, dist, flags=0)
        
        if (s == False):
            continue
        
        srvec = 'rvec:' + str(rvec[0][0])[:5] + " " + str(rvec[1][0])[:5] + " " + str(rvec[2][0])[:5] + " "
        image = cv2.putText(frame, srvec, (50, 50), cv2.FONT_HERSHEY_PLAIN, 2, (0, 0, 255), 2, cv2.LINE_AA)
        
        stvec = 'tvec:' + str(tvec[0][0])[:5] + " " + str(tvec[1][0])[:5] + " " + str(tvec[2][0])[:5] + " "
        image = cv2.putText(frame, stvec, (50, 100), cv2.FONT_HERSHEY_PLAIN, 2, (0, 0, 255), 2, cv2.LINE_AA)
    
    #Сonverting vectors to board rotation angles
        Rt = cv2.Rodrigues(rvec)
        Rt = np.transpose(Rt[0])
        sy = math.sqrt(Rt[0, 0] * Rt[0, 0] + Rt[1, 0] * Rt[1, 0])
        singular = sy < 1e-6

        # rotation matrix to Euler Angles
        if not singular:
            x = math.atan2(Rt[2,1] , Rt[2,2])* (180 / np.pi)
            y = math.atan2(-Rt[2,0], sy)* (180 / np.pi)
            z = math.atan2(Rt[1,0], Rt[0,0])* (180 / np.pi)

        else:
            x = math.atan2(-Rt[1,2], Rt[1,1])* (180 / np.pi)
            y = math.atan2(-Rt[2,0], sy)* (180 / np.pi)
            z = 0

        image = cv2.putText(frame, str(x)[:5] + " " +
                                   str(y)[:5] + " " +
                                   str(z)[:5], (50, 150), cv2.FONT_HERSHEY_PLAIN, 2, (0, 0, 255), 2, cv2.LINE_AA)
    
    cv2.namedWindow('custom window', cv2.WINDOW_KEEPRATIO)
    cv2.imshow('custom window', frame)
    cv2.resizeWindow('custom window', 1280, 720)
    
    
    key = cv2.waitKey(120) & 0xFF
    
    if (key == ord('q')):
        break

cam.release()
cv2.destroyAllWindows()
cv2.waitKey(10)

In [ ]:
import numpy as np
import math

# A - matrix of intrinsic parameters
# a - angle of rotation of the camera around vertical axis
# b - inclination angle of the camera
# x, y - coordinates of the object in the picture
# h - height of the camera above the ground

def pic2r(A, alpha, beta, x, y, h):
    fx = A[0, 0]
    fy = A[1, 1]
    cx = A[0, 2]
    cy = A[1, 2]
    
    y_ = h / math.tan( beta + math.atan((y - cy) / fy))
    x_ = (y_**2 + h**2)**0.5 * (x - cx) / fx
    
    R = np.array([[math.cos(alpha), -math.sin(alpha)],
                  [math.sin(alpha), math.cos(alpha)]])
    
    rotated = R @ np.array([x_, y_])
    
    return rotated[0], rotated[1]

A = np.array(mtx)
dist = np.array(dist)

#print(pic2r(A, 0.3, 0.45, 10, 30, 10))

import numpy as np
import cv2

cam = cv2.VideoCapture(1, cv2.CAP_DSHOW)
cam.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cam.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)

x, y = 0, 0

while(True):
    success, frame = cam.read()
    
    if (success == False):
        print("Cannot read frame. Exiting")
        break
    
    blurred = cv2.blur(frame, (7, 7))
    hsv = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)
    
    mask = cv2.inRange(hsv, (15, 112, 92), (57, 255, 255))
    
    cv2.imshow("mask", mask)
    
    connectivity = 4
    output = cv2.connectedComponentsWithStats(mask, connectivity, cv2.CV_32S)
    
    num_labels = output[0]
    labels = output[1]
    stats = output[2]
    
    filtered = np.zeros_like(mask)
    
    obj_h = 0
    
    for i in range(1, num_labels):
        a = stats[i, cv2.CC_STAT_AREA]
        t = stats[i, cv2.CC_STAT_TOP]
        l = stats[i, cv2.CC_STAT_LEFT]
        w = stats[i, cv2.CC_STAT_WIDTH]
        h = stats[i, cv2.CC_STAT_HEIGHT]
        
        if (a >= 4500):
            cv2.rectangle(frame, (l, t), (l + w, t + h), (123, 223, 134), 3)
            
            x = l + w // 2
            y = t + h
            
            break
    
    xr, yr = pic2r(A, 0, 35 / 180 * math.pi, x, y, 46.5)
    
    image = cv2.putText(frame, str(xr)[:5] + " " + str(yr)[:5], (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 
                           1.5, (255, 255, 0), 1, cv2.LINE_AA)
    
    cv2.imshow("frame", frame)

    key = cv2.waitKey(90) & 0xFF
        
    if (key == ord('q')):
        break

cam.release()
cv2.destroyAllWindows()
cv2.waitKey(10)